# Assignment 06 - Notebook 05: Đóng Gói Triển Khai (Deployment) & Hệ Thống Dự Báo Tương Tác

**Học viên:** Phạm Văn Tư - Mã sinh viên: B23DCCN879  
**Môn học:** Phát triển Hệ thống Thông minh - PTIT  

Notebook này kiểm thử toàn diện quy trình đóng gói suy luận (Inference Pipeline) và triển khai ứng dụng web tương tác:
1. Nạp độc lập các file trọng số PyTorch (`.pth`) và Keras (`.keras`) đã lưu tại `models/`.
2. Chạy thử nghiệm suy luận dự báo giá cổ phiếu Amazon ngày kế tiếp từ cửa sổ 30 phiên.
3. Chạy thử nghiệm đánh giá rủi ro khách hàng rời bỏ từ chuỗi tương tác 12 tháng.
4. Tích hợp và kiểm thử API RESTful của Web App triển khai tại cổng `http://127.0.0.1:8080`.


In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import tensorflow as tf
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
torch.set_num_threads(1)

print(f"PyTorch: {torch.__version__}, TensorFlow: {tf.__version__}")

BASE_DIR = os.path.dirname(os.path.abspath(''))
MODEL_DIR = os.path.join(BASE_DIR, 'models')
DATA_DIR = os.path.join(BASE_DIR, 'data', 'processed')


PyTorch: 2.3.0, TensorFlow: 2.21.0


## 1. Tải Lại Mô Hình PyTorch Từ File Trọng Số `.pth`


In [2]:
class PyTorchRNN(nn.Module):
    def __init__(self, cell_type, input_dim=8, hidden_dim=64, output_dim=1, num_layers=1, bidirectional=False):
        super().__init__()
        self.cell_type = cell_type
        num_directions = 2 if bidirectional else 1
        if cell_type == 'rnn':
            self.rnn = nn.RNN(input_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        elif cell_type == 'lstm':
            self.rnn = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers, batch_first=True, bidirectional=bidirectional)
        elif cell_type == 'gru':
            self.rnn = nn.GRU(input_dim, hidden_dim, num_layers=num_layers, batch_first=True, bidirectional=bidirectional)
            
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * num_directions, 32),
            nn.ReLU(),
            nn.Linear(32, output_dim)
        )
        
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])

# Nạp mô hình PyTorch LSTM dự báo cổ phiếu
stock_model = PyTorchRNN('lstm', input_dim=8, hidden_dim=64, output_dim=1)
stk_ckpt = os.path.join(MODEL_DIR, 'pytorch_amzn_lstm.pth')
stock_model.load_state_dict(torch.load(stk_ckpt, weights_only=True, map_location='cpu'))
stock_model.eval()
print(f"✓ Nạp thành công mô hình PyTorch Stock LSTM từ: {stk_ckpt}")

# Nạp mô hình PyTorch Customer Churn
cust_model = PyTorchRNN('lstm', input_dim=8, hidden_dim=64, output_dim=2)
cst_ckpt = os.path.join(MODEL_DIR, 'pytorch_cust_lstm.pth')
cust_model.load_state_dict(torch.load(cst_ckpt, weights_only=True, map_location='cpu'))
cust_model.eval()
print(f"✓ Nạp thành công mô hình PyTorch Customer LSTM từ: {cst_ckpt}")


✓ Nạp thành công mô hình PyTorch Stock LSTM từ: /Users/phamvantu/Desktop/INTEL_SYS/src/Assigment 06/models/pytorch_amzn_lstm.pth
✓ Nạp thành công mô hình PyTorch Customer LSTM từ: /Users/phamvantu/Desktop/INTEL_SYS/src/Assigment 06/models/pytorch_cust_lstm.pth


## 2. Kiểm Thử Suy Luận Dự Báo Cổ Phiếu Amazon
Lấy 30 phiên giao dịch cuối cùng để dự đoán giá đóng cửa tiếp theo.


In [3]:
stk_data = np.load(os.path.join(DATA_DIR, 'stock_amazon_processed.npz'))
X_sample = stk_data['X_test'][-1:] # 1 mẫu (1, 30, 8)
c_min, c_max = float(stk_data['close_min']), float(stk_data['close_max'])
y_actual = float(stk_data['y_test'][-1] * (c_max - c_min) + c_min)

with torch.no_grad():
    t0 = time.perf_counter()
    pred_norm = stock_model(torch.from_numpy(X_sample).float()).item()
    infer_time = (time.perf_counter() - t0) * 1000.0

pred_usd = pred_norm * (c_max - c_min) + c_min

print("=" * 55)
print("  KẾT QUẢ SUY LUẬN DỰ BÁO GIÁ CỔ PHIẾU AMAZON")
print("=" * 55)
print(f"Giá thực tế ngày mục tiêu : ${y_actual:.2f} USD")
print(f"Mô hình LSTM dự đoán      : ${pred_usd:.2f} USD")
print(f"Độ lệch tuyệt đối         : ${abs(pred_usd - y_actual):.2f} USD")
print(f"Thời gian suy luận        : {infer_time:.2f} ms")
print("=" * 55)


  KẾT QUẢ SUY LUẬN DỰ BÁO GIÁ CỔ PHIẾU AMAZON
Giá thực tế ngày mục tiêu : $221.30 USD
Mô hình LSTM dự đoán      : $179.91 USD
Độ lệch tuyệt đối         : $41.39 USD
Thời gian suy luận        : 19.56 ms


## 3. Kiểm Thử Suy Luận Đánh Giá Rủi Ro Khách Hàng Rời Bỏ (Churn)


In [4]:
cst_data = np.load(os.path.join(DATA_DIR, 'customer_churn_processed.npz'))
X_cust_sample = cst_data['X_test'][:2] # 2 mẫu
y_cust_actual = cst_data['y_test'][:2]

with torch.no_grad():
    t0 = time.perf_counter()
    logits = cust_model(torch.from_numpy(X_cust_sample).float())
    probs = torch.softmax(logits, dim=1).numpy()
    infer_time = (time.perf_counter() - t0) * 1000.0

print("=" * 55)
print("  KẾT QUẢ SUY LUẬN ĐÁNH GIÁ RỦI RO KHÁCH HÀNG RỜI BỎ")
print("=" * 55)
for i in range(len(X_cust_sample)):
    churn_prob = probs[i, 1] * 100
    status = "RỜI BỎ (CHURN)" if churn_prob > 50 else "GẮN BÓ (LOYAL)"
    truth = "RỜI BỎ (CHURN)" if y_cust_actual[i] == 1 else "GẮN BÓ (LOYAL)"
    print(f"Mẫu {i+1}: Thực tế = {truth} | Dự đoán = {status} (Xác suất Churn: {churn_prob:.1f}%)")
print(f"Thời gian suy luận trung bình: {infer_time/len(X_cust_sample):.2f} ms/mẫu")
print("=" * 55)


  KẾT QUẢ SUY LUẬN ĐÁNH GIÁ RỦI RO KHÁCH HÀNG RỜI BỎ
Mẫu 1: Thực tế = GẮN BÓ (LOYAL) | Dự đoán = GẮN BÓ (LOYAL) (Xác suất Churn: 0.0%)
Mẫu 2: Thực tế = GẮN BÓ (LOYAL) | Dự đoán = GẮN BÓ (LOYAL) (Xác suất Churn: 0.0%)
Thời gian suy luận trung bình: 1.48 ms/mẫu


## 4. Kiểm Thử Suy Luận Với Các Mô Hình Keras (.keras)


In [5]:
# Nạp và kiểm thử mô hình Keras Stock LSTM
keras_stock_path = os.path.join(MODEL_DIR, 'keras_amzn_lstm.keras')
k_stock_model = tf.keras.models.load_model(keras_stock_path)
t0 = time.perf_counter()
k_pred_norm = k_stock_model.predict(X_sample, verbose=0).item()
k_stock_time = (time.perf_counter() - t0) * 1000.0
k_pred_usd = k_pred_norm * (c_max - c_min) + c_min

print("=" * 55)
print("  KẾT QUẢ SUY LUẬN MÔ HÌNH KERAS LSTM (STOCK)")
print("=" * 55)
print(f"Giá thực tế ngày mục tiêu : ${y_actual:.2f} USD")
print(f"Mô hình Keras dự đoán     : ${k_pred_usd:.2f} USD")
print(f"Thời gian suy luận        : {k_stock_time:.2f} ms")
print("=" * 55)

# Nạp và kiểm thử mô hình Keras Customer Churn
keras_cust_path = os.path.join(MODEL_DIR, 'keras_cust_lstm.keras')
k_cust_model = tf.keras.models.load_model(keras_cust_path)
t0 = time.perf_counter()
k_probs = k_cust_model.predict(X_cust_sample, verbose=0)
k_cust_time = (time.perf_counter() - t0) * 1000.0

print()
print("=" * 55)
print("  KẾT QUẢ SUY LUẬN MÔ HÌNH KERAS LSTM (CUSTOMER CHURN)")
print("=" * 55)
for i in range(len(X_cust_sample)):
    churn_prob = k_probs[i, 1] * 100
    status = "RỜI BỎ (CHURN)" if churn_prob > 50 else "GẮN BÓ (LOYAL)"
    print(f"Mẫu {i+1}: Dự đoán = {status} (Xác suất Churn: {churn_prob:.1f}%)")
print(f"Thời gian suy luận: {k_cust_time/len(X_cust_sample):.2f} ms/mẫu")
print("=" * 55)


  KẾT QUẢ SUY LUẬN MÔ HÌNH KERAS LSTM (STOCK)
Giá thực tế ngày mục tiêu : $221.30 USD
Mô hình Keras dự đoán     : $192.77 USD
Thời gian suy luận        : 93.37 ms



  KẾT QUẢ SUY LUẬN MÔ HÌNH KERAS LSTM (CUSTOMER CHURN)
Mẫu 1: Dự đoán = GẮN BÓ (LOYAL) (Xác suất Churn: 0.0%)
Mẫu 2: Dự đoán = GẮN BÓ (LOYAL) (Xác suất Churn: 0.0%)
Thời gian suy luận: 32.77 ms/mẫu


## 5. Hướng Dẫn Vận Hành Web App Triển Khai
Web App được xây dựng hoàn chỉnh tại thư mục `src/Assigment 06/web_app/`.
Để khởi chạy ứng dụng tương tác trên trình duyệt:
```bash
# Cách 1: Sử dụng bash script
bash "src/Assigment 06/run_web_app.sh"

# Cách 2: Chạy trực tiếp qua Python
python3 "src/Assigment 06/web_app/app.py" --port 8080
```
Sau đó mở trình duyệt tại: `http://127.0.0.1:8080` để trải nghiệm trực quan cả 2 phân hệ.
